In [1]:
import numpy as np 
import pickle 
from scipy import stats
from tqdm.auto import tqdm



In [2]:
# Get model activation files 
from pathlib import Path 

paths = list(Path('attn_cue_models/').glob('*/*model_activations_0dB.npz'))

In [3]:
paths[0].parent.name

'attn_cue_match_target_speech_and_noise_coch_attn_only'

In [4]:
for model_path in paths:
    model_name = model_path.parent.name
    activations = np.load(model_path, allow_pickle=True)

    mixture_reps = activations['mixture_reps'].item()
    fg_reps = activations['fg_reps'].item()
    bg_reps = activations['bg_reps'].item()

    ## Get correlations by layer

    fg_corr_results = {}
    bg_corr_results = {}

    n_sounds = 100

    for layer, mixture_acts in tqdm(mixture_reps.items()):
        fg_acts  = fg_reps[layer]
        bg_acts = bg_reps[layer]
        
        # Calculate corr coefs for mixture and foreground
        fg_r = np.corrcoef(mixture_acts, fg_acts)
        # get coeffs of wanted samples
        fg_corr_results[layer] = np.diagonal(fg_r[:n_sounds, n_sounds:])
        
        # Calculate corr coefs for mixture and background
        bg_r = np.corrcoef(mixture_acts, bg_acts)
        # get coeffs of background 
        bg_corr_results[layer] = np.diagonal(bg_r[:n_sounds, n_sounds:])
        
    # Save results as dict
    out_name = f'{model_name}_corrs_at_0dB.pkl'

    out_dict = dict(fg_corr_results=fg_corr_results, bg_corr_results=bg_corr_results)

    with open(out_name ,'wb') as f:
        pickle.dump(out_dict, f)

  0%|          | 0/9 [00:00<?, ?it/s]

/home/imgriff/.local/lib/python3.9/site-packages/numpy/lib/function_base.py:2853: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/imgriff/.local/lib/python3.9/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


  0%|          | 0/9 [00:00<?, ?it/s]